# backwardCGM-PD — Air Quality trên Kaggle

Notebook độc lập để fit lại pdglasso và backwardCGM từ residual AR(1) mà tác giả cung cấp.

In [ ]:
import importlib.util, subprocess, sys
required = {
    "rdata": "rdata>=0.11", "networkx": "networkx>=3.0",
    "sklearn": "scikit-learn>=1.3", "joblib": "joblib>=1.3",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies: OK")

In [ ]:
from pathlib import Path
import json, shutil, zipfile
import pandas as pd
from IPython.display import Image, display

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/air-quality")
RESULTS = Path("/kaggle/working/air-quality-results")
RESULTS.mkdir(parents=True, exist_ok=True)

# Khôi phục checkpoint từ output của một Kaggle Version trước nếu đã Add Input.
checkpoint_archives = list(INPUT_ROOT.rglob("air-quality-results.zip"))
checkpoint_files = list(INPUT_ROOT.rglob("air-quality.json"))
if checkpoint_archives:
    with zipfile.ZipFile(checkpoint_archives[0]) as archive:
        archive.extractall(RESULTS)
    print("Restored checkpoint:", checkpoint_archives[0])
elif checkpoint_files:
    shutil.copytree(checkpoint_files[0].parent, RESULTS, dirs_exist_ok=True)
    print("Restored checkpoint:", checkpoint_files[0])

archives = list(INPUT_ROOT.rglob("backwardCGM-PD-kaggle-dataset.zip"))
scripts = list(INPUT_ROOT.rglob("python-port/experiments/air_quality.py"))
if archives:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT)
elif scripts:
    shutil.copytree(scripts[0].parents[2], WORK_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError("Hãy Add Input dataset backwardCGM-PD")

PORT_ROOT = WORK_ROOT / "python-port"
AIR_DATA = WORK_ROOT / "applications/airquality/airdata/airdataAR1.RData"
if not (PORT_ROOT / "experiments/air_quality.py").exists() or not AIR_DATA.exists():
    raise FileNotFoundError("Dataset Air Quality không đầy đủ")
print("Dataset: OK\nInput :", AIR_DATA, "\nOutput:", RESULTS)

## Cấu hình

Notebook mặc định chạy full: pdglasso dùng 10 điểm tuyến tính mỗi lambda grid như pdRCON.fit 1.0.0 và chạy thêm backward twin-lattice.

In [ ]:
FULL_EXPERIMENT = True
GRID_POINTS = 10 if FULL_EXPERIMENT else 2
print({"full": FULL_EXPERIMENT, "grid_points": GRID_POINTS})

In [ ]:
output = RESULTS / "air-quality.json"
command = [
    sys.executable, "-u", str(PORT_ROOT / "experiments/air_quality.py"),
    "--data", str(AIR_DATA), "--grid-points", str(GRID_POINTS),
    "--alpha", "0.05", "--itmax", "1000", "--output", str(output), "--resume", "--verbose",
]
if not FULL_EXPERIMENT:
    command.append("--skip-backward")
print("Running:", " ".join(command))
subprocess.run(command, cwd=PORT_ROOT, check=True)

In [ ]:
result = json.loads(output.read_text(encoding="utf-8"))
rows = []
for name in ("pdglasso_covariance", "pdglasso_correlation"):
    item = result[name]
    rows.append({
        "method": name, "lambda1": item["best_lambdas"][0],
        "lambda2": item["best_lambdas"][1],
        "LRT p-value": item["lrt_pvalue_on_unscaled_data"],
        "edges": len(item["model"]["E"]),
    })
display(pd.DataFrame(rows))
if "parity_with_r_artifacts" in result:
    parity_rows = [{"component": key, **value} for key, value in result["parity_with_r_artifacts"].items()]
    display(pd.DataFrame(parity_rows))
figure_dir = output.with_suffix("")
for name in ("variable-scales.png", "figure-9-matrix-models.png", "figure-9-python-refit.png", "pdglasso-covariance.png", "pdglasso-correlation.png", "backward.png"):
    figure = figure_dir / name
    if figure.exists():
        display(Image(filename=str(figure)))
archive = shutil.make_archive("/kaggle/working/air-quality-results", "zip", root_dir=RESULTS)
print("Download:", archive)

**Checkpoint:** JSON được ghi atomically sau từng giai đoạn: pdglasso-covariance, pdglasso-correlation và backwardCGM. Để tiếp tục trong session mới, hãy **Save Version** rồi **Add Input** output cũ; notebook tự khôi phục `air-quality-results.zip` hoặc `air-quality.json`.

**Blocker:** repo bắt đầu từ `airdataAR1.RData`; không có raw data và code tạo residual AR(1), nên notebook tái chạy phần model fitting chứ không tái tạo preprocessing end-to-end.

**Parity:** pdglasso Python đã được kiểm tra khớp lambda, selected graph và precision matrix của hai artifact R. Riêng `air.backward.RData` không tương thích với dữ liệu/code được commit: final graph có LRT p-value dưới `1e-30`, trong khi `backwardCGMpd(alpha=.05)` chỉ chấp nhận model có `p >= .05`. Vì vậy `figure-9-matrix-models.png` dựng từ artifact R để khớp paper; `figure-9-python-refit.png` là kết quả refit độc lập và được giữ riêng.